In [ ]:
#!/usr/bin/env python3
"""
Analyze and print statistics from JSON result files in a specified directory.
"""

import json
import os
import argparse
from collections import Counter, defaultdict
from pathlib import Path

# Load exclusion lists
exclude_ids_by_task = {
    "vague": ["3I02618YA14S7SHVOPAYBZTIDARUPP", "3KIBXJ1WD6SWJW0IFBTHGCFU1SKOKK", "3NPI0JQDAP3D7F26OKKO637GUJPPT8", "3SITXWYCNW7IK2AGAP3K0MNXQ3DBXR"],
    "reverse": ["39L1G8WVWRP5R6LAO337NULKX9931E"],
    "biased": [],
    "fake": ["9929"]
}

def load_results_from_directory(results_dir, task, resp_flag):
    results = []
    json_files = list(Path(results_dir).glob("result_*.json"))
    
    print(f"all {len(json_files)} files found in {results_dir}")
    
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                results.append(data)
        except Exception as e:
            print(f"Error loading {json_file}: {e}")

    # Exclude results based on task-specific criteria
    if resp_flag:
        if task in exclude_ids_by_task:
            exclude_ids = set(exclude_ids_by_task[task])
            results = [r for r in results if r.get('sample_id') not in exclude_ids]
    
    return results


def analyze_assessments(results):
    assessments = []
    for result in results:
        parsed_response = result.get('parsed_response', {})
        assessment = parsed_response.get('assessment', 'unknown')
        assessments.append(assessment)
    
    assessment_counts = Counter(assessments)
    return assessment_counts


def analyze_violated_categories(results):
    all_categories = []
    category_combinations = []
    
    for result in results:
        parsed_response = result.get('parsed_response', {})
        violated_categories = parsed_response.get('violated_categories', [])
        
        # record single category
        for category in violated_categories:
            all_categories.append(category)
        
        # record category combination
        if violated_categories:
            combo = sorted(violated_categories)
            category_combinations.append(tuple(combo))
        else:
            category_combinations.append(('None',))
    
    single_category_counts = Counter(all_categories)
    combination_counts = Counter(category_combinations)
    
    return single_category_counts, combination_counts


def print_statistics(results):
    print(f"{'='*60}")
    print(f"all {len(results)} samples")
    print(f"{'='*60}")
    
    # 1. Assessment Distribution
    assessment_counts = analyze_assessments(results)
    print("Assessment Distribution:")
    print("-" * 30)
    total_samples = sum(assessment_counts.values())
    for assessment, count in assessment_counts.most_common():
        percentage = (count / total_samples) * 100
        print(f"  {assessment:10}: {count:4} ({percentage:5.1f}%)")
    
    # 2. Violated Categories
    single_counts, combo_counts = analyze_violated_categories(results)
    
    print("Category :")
    print("-" * 30)
    if single_counts:
        for category, count in single_counts.most_common():
            print(f"  {category}: {count} ({(count / total_samples) * 100:.1f}%)")
    else:
        print("no unsafe categories found.")
    
    print("Category combinations :")
    print("-" * 40)
    for combo, count in combo_counts.most_common(10):
        combo_str = ", ".join(combo) if combo != ('None',) else "safe"
        print(f"  {combo_str}: {count}")



In [ ]:
for model in ["aegis_d_ori_resp", "aegis_p_ori_resp", "aegis_p_cus_resp", "aegis_d_cus_resp", "aegis_d_ori", "aegis_d_cus", "aegis_p_cus", "aegis_p_ori"]:
    for task in ["vague", "reverse", "biased", "fake"]:
        results_dir=f'./results_{task}_{model}'

        if not os.path.exists(results_dir):
            print(f"Err: dir {results_dir} not exists.")

        if model.endswith("resp"):
            resp_flag = True
        else:
            resp_flag = False

        results = load_results_from_directory(results_dir, task, resp_flag)

        if not results:
            print("Err: no valid result files found.")

        print_statistics(results)
        print("\n\n")

all 2500 files found in ./results_vague_p_cus_resp
all 2496 samples
Assessment Distribution:
------------------------------
  safe      : 1467 ( 58.8%)
  unsafe    : 1029 ( 41.2%)
Category :
------------------------------
  O1: 608 (24.4%)
  O10: 216 (8.7%)
  O9: 80 (3.2%)
  O4: 53 (2.1%)
  O3: 47 (1.9%)
  O2: 22 (0.9%)
  O5: 2 (0.1%)
  O8: 1 (0.0%)
Category combinations :
----------------------------------------
  safe: 1467
  O1: 608
  O10: 216
  O9: 80
  O4: 53
  O3: 47
  O2: 22
  O5: 2
  O8: 1



all 2500 files found in ./results_reverse_p_cus_resp
all 2499 samples
Assessment Distribution:
------------------------------
  unsafe    : 1940 ( 77.6%)
  safe      :  559 ( 22.4%)
Category :
------------------------------
  O1: 1448 (57.9%)
  O10: 392 (15.7%)
  O9: 35 (1.4%)
  O4: 30 (1.2%)
  O2: 17 (0.7%)
  O5: 9 (0.4%)
  O3: 7 (0.3%)
  O8: 1 (0.0%)
  O7: 1 (0.0%)
Category combinations :
----------------------------------------
  O1: 1448
  safe: 559
  O10: 392
  O9: 35
  O4: 30
  O2: 1